# Can two identical fiber networks respond differently to the same load?

Run this CPU-only example to compare three material layouts in the **same 3D bonded fiber network**. The geometry and electrical conductivity start out identical. We apply the same signed grip forces, then measure how resistance changes as each network deforms.

**Browser:** upload this file in [Google Colab](https://colab.research.google.com/) and choose **Runtime → Run all**. **Jupyter:** open the file in a Python 3 kernel and run the cells in order. No data files, GPU or clone are required. The first two cells download the published package and one pinned public example.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

required = "4.25.0"
try:
    installed = version("fibernet")
except PackageNotFoundError:
    installed = None
if installed != required:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"fibernet=={required}"])
assert version("fibernet") == required

## 1. Load the reproducible model

The [MIT-licensed example](https://github.com/GellmanSparrowS/fibernet/blob/c0a3f5f9212c951f628ef3fca801e36c549110b1/examples/tet4_mechanoelectric_looped_network.py) is pinned to a GitHub commit and checked against a SHA-256 fingerprint before running. It uses FiberNet's native solid and DC solvers.

In [ ]:
from hashlib import sha256
from pathlib import Path
import runpy
from tempfile import TemporaryDirectory
from urllib.request import urlopen

base = "https://raw.githubusercontent.com/GellmanSparrowS/fibernet/"
revision = "c0a3f5f9212c951f628ef3fca801e36c549110b1"
path = "examples/tet4_mechanoelectric_looped_network.py"
with urlopen(f"{base}{revision}/{path}", timeout=25) as response:
    source = response.read()
expected = "11e6af6db5952909f8cbf2bea4375a5f64a90f2efb86a448e49ec53f1c97c15b"
if sha256(source).hexdigest() != expected:
    raise ValueError("Source fingerprint mismatch; the example was not executed.")
with TemporaryDirectory() as folder:
    script = Path(folder) / "network.py"
    script.write_bytes(source)
    example = runpy.run_path(str(script))

points, tets = example["_lattice"]()
print(f"{len(points):,} vertices, {len(tets):,} tetrahedra")

## 2. Keep the applied force the same

All three cases use the same grip constraints; their axial strokes differ to match the forces. The solver checks reactions of **−0.5 N** and **+0.5 N** and recomputes resistance on the deformed geometry. We compare uniform 1 GPa, uniform 2 GPa, and one axial half at 2 GPa. A smaller slope means less fractional resistance change per newton, not lower resistance at rest.

In [ ]:
import numpy as np

data = example["run_matched_force"]()
print(f"{'Layout':<16} {'R₀ (Ω)':>12} {'(1/R₀) dR/dF (1/N)':>22}")
for name, item in data.items():
    assert np.isclose(item.compressive_reaction_N, -0.5)
    assert np.isclose(item.tensile_reaction_N, 0.5)
    print(f"{name:<16} {item.rest_resistance_ohm:>12.3f} {item.resistance_slope_per_N:>22.6g}")
rest = [item.rest_resistance_ohm for item in data.values()]
np.testing.assert_allclose(rest, rest[0], rtol=1e-10)
assert data["uniform_stiff"].resistance_slope_per_N < data["stiff_half"].resistance_slope_per_N < data["uniform"].resistance_slope_per_N

## 3. See the network and the result

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from itertools import combinations

faces = np.sort(tets[:, list(combinations(range(4), 3))].reshape(-1, 3), axis=1)
faces, counts = np.unique(faces, axis=0, return_counts=True)
surface = faces[counts == 1]
edges = np.unique(np.sort(surface[:, [(0, 1), (0, 2), (1, 2)]].reshape(-1, 2), axis=1), axis=0)
fig = plt.figure(figsize=(11, 4.6), constrained_layout=True)
mesh_ax = fig.add_subplot(1, 2, 1, projection="3d")
mesh_ax.add_collection3d(Line3DCollection(points[edges], colors="#177667", linewidths=.5, alpha=.65))
mesh_ax.auto_scale_xyz(*points.T)
mesh_ax.set_box_aspect(np.ptp(points, axis=0))
mesh_ax.view_init(elev=24, azim=-60)
mesh_ax.set_title("One network, three material layouts", loc="left", pad=10)
mesh_ax.set(xlabel="x (mm)", ylabel="y (mm)")
mesh_ax.set_yticks([.5, 1., 1.5])
mesh_ax.set_zticks([.5, 1., 1.5])

ax = fig.add_subplot(1, 2, 2)
labels = ["Uniform 1 GPa", "Uniform 2 GPa", "Half stiffened"]
slopes = [100 * item.resistance_slope_per_N for item in data.values()]
bars = ax.barh(labels, slopes, color=["#177667", "#517782", "#D18754"])
ax.invert_yaxis()
ax.set_xlim(0, max(slopes)*1.22)
ax.bar_label(bars, fmt="%.3f", padding=4)
ax.set(xlabel="Relative resistance slope (% per N)", title="At matched grip forces")
for side in ("right", "top"):
    ax.spines[side].set_visible(False)
ax.grid(axis="x", alpha=.2)
ax.set_axisbelow(True)
plt.show()

## What did we learn?

Stiffening the solid changes how much it deforms under the same force, even though its starting resistance is unchanged. This reproduces a geometry-only, small-deformation effect on a synthetic network, not intrinsic piezoresistivity or electrical feedback into mechanics.

Try the [other 3D examples](https://github.com/GellmanSparrowS/fibernet/tree/main/examples) next. If you try FiberNet on your own network, tell us what input or workflow was missing in [GitHub Issues](https://github.com/GellmanSparrowS/fibernet/issues). If you found this useful, a [GitHub Star](https://github.com/GellmanSparrowS/fibernet) helps others find it.